In [17]:
#!pip install pandas numpy scikit-learn matplotlib seaborn

In [14]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer
import warnings
warnings.filterwarnings('ignore')
import os

In [15]:
class ContentBasedRecommender:
    def __init__(self):
        self.movies_df = None
        self.ratings_df = None
        self.user_profiles = {}
        self.movie_features = None
        self.feature_matrix = None
        self.movie_similarity = None
        
    def download_movielens_data(self, data_dir='ml-100k'):
        """Download MovieLens 100k dataset if not already present"""
        
        if os.path.exists(data_dir):
            print(f"MovieLens data already exists in {data_dir}/")
            return data_dir
            
        print("Downloading MovieLens 100k dataset...")
        url = "http://files.grouplens.org/datasets/movielens/ml-100k.zip"
        zip_path = "ml-100k.zip"
        
        try:
            urllib.request.urlretrieve(url, zip_path)
            print("Download completed. Extracting...")
            
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall()
                
            os.remove(zip_path)  # Clean up zip file
            print(f"Data extracted to {data_dir}/")
            return data_dir
            
        except Exception as e:
            print(f"Error downloading data: {e}")
            print("Please manually download from: https://grouplens.org/datasets/movielens/100k/")
            return None
        
    def load_data(self, data_dir=None):
        """Load and preprocess MovieLens data"""
        
        # Download data if not present
        if data_dir is None:
            data_dir = self.download_movielens_data()
            if data_dir is None:
                raise FileNotFoundError("Could not download or find MovieLens data")
        
        movies_path = os.path.join(data_dir, 'u.item')
        ratings_path = os.path.join(data_dir, 'u.data')
        
        # Check if files exist
        if not os.path.exists(movies_path):
            raise FileNotFoundError(f"Movies file not found: {movies_path}")
        if not os.path.exists(ratings_path):
            raise FileNotFoundError(f"Ratings file not found: {ratings_path}")
        
        # Load movie data
        movie_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url'] + \
                     [f'genre_{i}' for i in range(19)]  # 19 genre columns
        
        self.movies_df = pd.read_csv(movies_path, sep='|', names=movie_cols, 
                                   encoding='latin-1', index_col=0)
        
        # Extract genres into a more usable format
        genre_names = ['unknown', 'Action', 'Adventure', 'Animation', 'Children', 
                      'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
                      'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance',
                      'Sci-Fi', 'Thriller', 'War', 'Western']
        
        # Create genre list for each movie
        self.movies_df['genres'] = self.movies_df[
            [f'genre_{i}' for i in range(19)]
        ].apply(lambda x: [genre_names[i] for i, val in enumerate(x) if val == 1], axis=1)
        
        # Extract year from title (format: "Movie Title (Year)")
        self.movies_df['year'] = self.movies_df['title'].str.extract(r'\((\d{4})\)')
        self.movies_df['year'] = pd.to_numeric(self.movies_df['year'], errors='coerce')
        
        # Clean title (remove year)
        self.movies_df['clean_title'] = self.movies_df['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)
        
        # Load ratings data
        self.ratings_df = pd.read_csv(ratings_path, sep='\t', 
                                    names=['user_id', 'movie_id', 'rating', 'timestamp'])
        
        print(f"Loaded {len(self.movies_df)} movies and {len(self.ratings_df)} ratings")
        
    def create_movie_features(self):
        """Create feature vectors for movies based on genres"""
        
        # Convert genres to binary matrix
        mlb = MultiLabelBinarizer()
        genre_matrix = mlb.fit_transform(self.movies_df['genres'])
        genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_, 
                               index=self.movies_df.index)
        
        # Add decade feature (normalized)
        decades = (self.movies_df['year'] // 10 * 10).fillna(1990)  # Default to 1990s
        decade_dummies = pd.get_dummies(decades, prefix='decade')
        
        # Ensure decade_dummies has the same index as genre_df
        decade_dummies.index = genre_df.index
        
        # Combine features and ensure all values are numeric
        self.feature_matrix = pd.concat([genre_df, decade_dummies], axis=1).fillna(0)
        
        # Explicitly convert to float to avoid dtype issues
        self.feature_matrix = self.feature_matrix.astype(float)
        
        print(f"Created feature matrix with {self.feature_matrix.shape[1]} features")
        return self.feature_matrix
        
    def build_user_profile(self, user_id):
        """Build a user's content profile based on their rating history"""
        
        # Get user's ratings
        user_ratings = self.ratings_df[self.ratings_df['user_id'] == user_id]
        
        if len(user_ratings) == 0:
            return None
            
        # Weight movies by rating (give more importance to highly rated movies)
        # Only consider movies rated 4 or 5 as "liked"
        liked_movies = user_ratings[user_ratings['rating'] >= 4]
        
        if len(liked_movies) == 0:
            # If no highly rated movies, use all ratings above average
            avg_rating = user_ratings['rating'].mean()
            liked_movies = user_ratings[user_ratings['rating'] >= avg_rating]
        
        # Get feature vectors for liked movies
        liked_movie_features = self.feature_matrix.loc[liked_movies['movie_id']]
        
        # Create weighted average profile (weight by rating)
        weights = liked_movies.set_index('movie_id')['rating']
        weighted_features = liked_movie_features.multiply(weights, axis=0)
        user_profile = weighted_features.mean()
        
        self.user_profiles[user_id] = user_profile
        return user_profile
        
    def get_recommendations(self, user_id, n_recommendations=10):
        """Get movie recommendations for a user"""
        
        # Build user profile if not exists
        if user_id not in self.user_profiles:
            user_profile = self.build_user_profile(user_id)
            if user_profile is None:
                return pd.DataFrame()  # Return empty if no ratings
        else:
            user_profile = self.user_profiles[user_id]
            
        # Get movies the user hasn't seen
        user_movies = set(self.ratings_df[self.ratings_df['user_id'] == user_id]['movie_id'])
        unseen_movies = self.feature_matrix.index.difference(user_movies)
        
        if len(unseen_movies) == 0:
            return pd.DataFrame()
            
        # Calculate similarity between user profile and unseen movies
        unseen_features = self.feature_matrix.loc[unseen_movies]
        similarities = cosine_similarity([user_profile], unseen_features)[0]
        
        # Create recommendations dataframe
        recommendations = pd.DataFrame({
            'movie_id': unseen_movies,
            'similarity_score': similarities
        })
        
        # Add movie information
        recommendations = recommendations.merge(
            self.movies_df[['clean_title', 'genres', 'year']], 
            left_on='movie_id', right_index=True
        )
        
        # Sort by similarity and return top N
        recommendations = recommendations.sort_values('similarity_score', ascending=False)
        return recommendations.head(n_recommendations)
        
    def explain_recommendation(self, user_id, movie_id):
        """Explain why a movie was recommended"""
        
        if user_id not in self.user_profiles:
            self.build_user_profile(user_id)
            
        user_profile = self.user_profiles[user_id]
        movie_features = self.feature_matrix.loc[movie_id]
        
        # Ensure both are numeric and aligned
        user_profile = pd.to_numeric(user_profile, errors='coerce').fillna(0)
        movie_features = pd.to_numeric(movie_features, errors='coerce').fillna(0)
        
        # Find the most important features
        feature_contributions = user_profile * movie_features
        
        # Convert to numeric explicitly and get top features
        feature_contributions = pd.to_numeric(feature_contributions, errors='coerce').fillna(0)
        top_features = feature_contributions.nlargest(5)
        
        movie_info = self.movies_df.loc[movie_id]
        
        print(f"\nWhy '{movie_info['clean_title']}' was recommended:")
        print(f"Overall similarity score: {cosine_similarity([user_profile], [movie_features])[0][0]:.3f}")
        print("\nTop matching features:")
        for feature, score in top_features.items():
            if score > 0:
                print(f"  - {feature}: {score:.3f}")
                
    def get_user_profile(self, user_id):
        """Analyze and summarize a user's taste preferences"""
        
        if user_id not in self.user_profiles:
            self.build_user_profile(user_id)
            
        user_profile = self.user_profiles[user_id]

        return user_profile

In [16]:
recommender = ContentBasedRecommender()

In [17]:
print("Loading MovieLens data...")
recommender.load_data()

Loading MovieLens data...
MovieLens data already exists in ml-100k/
Loaded 1682 movies and 100000 ratings


In [18]:
print("\nCreating movie feature vectors...")
recommender.create_movie_features()


Creating movie feature vectors...
Created feature matrix with 27 features


,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,...,Western,unknown,decade_1920.0,decade_1930.0,decade_1940.0,decade_1950.0,decade_1960.0,decade_1970.0,decade_1980.0,decade_1990.0
movie_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
5,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1678,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1679,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1680,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [19]:
test_user = 196

In [20]:
print(f"\nAnalyzing user {test_user}'s preferences...")
pd.DataFrame(recommender.get_user_profile(test_user)).T


Analyzing user 196's preferences...


,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,...,Western,unknown,decade_1920.0,decade_1930.0,decade_1940.0,decade_1950.0,decade_1960.0,decade_1970.0,decade_1980.0,decade_1990.0
0,0.0,0.227273,0.0,0.227273,3.545455,0.0,0.181818,1.5,0.181818,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.590909,0.636364,3.136364


In [21]:
print(f"\nGenerating recommendations for user {test_user}...")
recommendations = recommender.get_recommendations(test_user, n_recommendations=2)
if not recommendations.empty:
    print(f"\nTop 2 Content-Based Recommendations:")
    print("="*60)
    for idx, row in recommendations.iterrows():
        genres_str = ', '.join(row['genres'][:3])  # Show first 3 genres
        print(f"{idx+1:2d}. {row['clean_title']} ({row['year']:.0f})")
        print(f"    Genres: {genres_str}")
        print(f"    Similarity: {row['similarity_score']:.3f}")
        print()
        
    # Explain a specific recommendation
    top_recommendation = recommendations.iloc[0]['movie_id']
    recommender.explain_recommendation(test_user, top_recommendation)
else:
    print("No recommendations found for this user.")


Generating recommendations for user 196...

Top 2 Content-Based Recommendations:
1322. Search for One-eye Jimmy, The (1996)
    Genres: Comedy
    Similarity: 0.908

1168. Amos & Andrew (1993)
    Genres: Comedy
    Similarity: 0.908


Why 'Search for One-eye Jimmy, The' was recommended:
Overall similarity score: 0.908

Top matching features:
  - Comedy: 3.545
  - decade_1990.0: 3.136
